In [1]:
import PeterChurchillFunctions as Function
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf
import statsmodels.api as sm
from scipy.odr import ODR, Model, RealData
from matplotlib.colors import LogNorm

In [4]:
ECPath = "/share/sabl0586/all_stations_EC-Earth_PRCP2SZDST_ilevall_levs_4Peter.nc"
ds = xr.open_dataset(ECPath, engine="netcdf4")#, chunks={'station': -1,
                                     #'time': 1000,      # split time into many parallel chunks
                                     #'lev': -1          # keep vertical levels moderately sized})
                                    # })
stations = ds["station"].values
TestStation = ['SMR-II']
Pressure = ds["pressure"]

# Compute the mean pressure across time and stations
# This gives you a 1D array of 34 levels
Pressure_mean_levs = Pressure.mean(dim=["time", "station"])#.load()

# Optionally, convert to hPa
Pressure_mean_levs = Pressure_mean_levs 

# Check result
print("Mean pressure levels:", Pressure_mean_levs.shape)
Pressure_mean_levs
radii = np.arange(20,51)
VarList = []
IFSVarList = []
x = xr.DataArray(np.logspace(-0.5,8, num=200), dims =['D'], coords= {'D':np.logspace(-0.5,8, num=200)})

Mean pressure levels: (34,)


In [3]:
def ECearthExtract_Dask(ECPath, station):#, chunks="auto"):
    """
    Lazily load EC-Earth data using Dask and optionally compute PNSD.

    Parameters
    ----------
    ECPath : str
        Path to EC-Earth NetCDF file.
    station : str
        Station name or coordinate.
    chunks : str or dict, optional
        Dask chunking specification. Default 'auto'.
    """

    import xarray as xr

    #Open lazily with Dask
    Data = xr.open_dataset(ECPath, engine="netcdf4")#, chunks=chunks)
    Data = Data.sel(station=station)

    ds = xr.Dataset()
    PNSD_ds = xr.Dataset()
    cdnc = xr.Dataset()
    ds_IFS = xr.Dataset()


    #Define variables for the ERF
    radius_variables = ['RDRY_NUS', 'RDRY_AIS', 'RDRY_ACS', 'RWET_AII', 'RDRY_COS', 'RWET_ACI', 'RWET_COI']
    Numb_variables = ['N_NUS', 'N_AIS', 'N_ACS', 'N_AII', 'N_COS', 'N_ACI', 'N_COI']
    ModesSigma = [1.59, 1.59, 1.59, 2.0, 1.59, 1.59, 2.0]
    
    for radius, number in zip(radius_variables, Numb_variables):
        if radius in Data and number in Data:
            ds[radius] = Data[radius]*1e9
            ds[radius].attrs["units"] = "nm"
            #print(f' {radius} added to Dataset')
            ds[number] = Data[number]/1e6
            ds[number].attrs["units"] = "#cm-3"
            #print(f' {number} added to Dataset')
        else:
            print(f'{radius, number} not found in EC Path Data')
            
    # ADD PRESSURE TO DATASET        
    ds['pressure'] = Data['pressure']
    ds['lev'] = ds['pressure'].mean('time')

    #Converting the IFS levels to match the EC-Earth Levs
    for ifs in [['var20', 'var22', 'var54']]:
        ds_IFS[ifs] = Data[ifs]
    ds_IFS = ds_IFS[['var20', 'var22', 'var54']].isel(lev=0).drop_vars('lev')
    ds_IFS['lev_ifs'] = ds_IFS['var54'].mean('time')

    
    return ds, ds_IFS

In [ ]:
# Open dataset to access pressure metadata and data lazily
Pressure_ds = xr.open_dataset(ECPath, engine="netcdf4")

# Extract the pressure variable (do not load all data yet)
Pressure = Pressure_ds["pressure"]

# Compute the mean pressure across time and stations
# This gives you a 1D array of 34 levels
Pressure_mean_levs = Pressure.mean(dim=["time", "station"])#.load()

# Optionally, convert to hPa
Pressure_mean_levs = Pressure_mean_levs / 100
Pressure_mean_levs.attrs["units"] = "hPa"

# Check result
print("Mean pressure levels:", Pressure_mean_levs.shape)
Pressure_mean_levs